# AI-Powered Skin Cancer Detection using Hybrid CNN + MobileNetV2
**Objective:** Build a production-quality skin cancer classification model using the HAM10000 dataset with a memory-efficient preprocessing pipeline.

In [ ]:
# CELL 1: Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import glob
import os
import pickle
import gc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, GlobalAveragePooling2D, Dense, Dropout, Input, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

In [ ]:
# CELL 2: Load metadata CSV
# Update path as per your Colab environment
metadata_path = 'HAM10000_metadata.csv'
df = pd.read_csv(metadata_path)

print("Metadata Shape:", df.shape)
display(df.head())
print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# CELL 3: Combine image folders and create image path dictionary
image_paths = {}
for x in glob.glob(os.path.join('HAM10000_images_part_1', '*.jpg')):
    image_paths[os.path.splitext(os.path.basename(x))[0]] = x

for x in glob.glob(os.path.join('HAM10000_images_part_2', '*.jpg')):
    image_paths[os.path.splitext(os.path.basename(x))[0]] = x

print(f"Total images found: {len(image_paths)}")

In [ ]:
# CELL 4: Create columns and Display class distribution
df['path'] = df['image_id'].map(image_paths.get)
df['cell_type'] = df['dx'] # dx is the class column in HAM10000

plt.figure(figsize=(10,6))
sns.countplot(x='cell_type', data=df, palette='viridis')
plt.title('Class Distribution')
plt.show()

print("Class Counts:\n", df['cell_type'].value_counts())

In [ ]:
# CELL 5: Label encoding
encoder = LabelEncoder()
df['encoded_label'] = encoder.fit_transform(df['cell_type'])

print("Encoded Classes:")
for i, label in enumerate(encoder.classes_):
    print(f"{i}: {label}")

In [ ]:
# CELL 6: Train-validation-test split (70-15-15)
# Split only dataframe to save memory
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['cell_type'])
val_df, test_df = train_test_split(test_df, test_size=0.5, random_state=42, stratify=test_df['cell_type'])

print(f"Train size: {len(train_df)}")
print(f"Val size: {len(val_df)}")
print(f"Test size: {len(test_df)}")

In [ ]:
# CELL 7: Create ImageDataGenerators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='path',
    y_col='cell_type',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_test_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='path',
    y_col='cell_type',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='path',
    y_col='cell_type',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
# CELL 8: Hybrid Model, Training, Evaluation, and Prediction

# 1. Compute class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['encoded_label']),
    y=train_df['encoded_label']
)
class_weights_dict = dict(enumerate(class_weights))

# 2. Build Hybrid Model using Functional API
input_layer = Input(shape=(224, 224, 3))

# --- Custom CNN Branch ---
x1 = Conv2D(32, (3,3), activation='relu', padding='same')(input_layer)
x1 = BatchNormalization()(x1)
x1 = MaxPooling2D((2,2))(x1)

x1 = Conv2D(64, (3,3), activation='relu', padding='same')(x1)
x1 = BatchNormalization()(x1)
x1 = MaxPooling2D((2,2))(x1)

x1 = Conv2D(128, (3,3), activation='relu', padding='same')(x1)
x1 = BatchNormalization()(x1)
x1 = MaxPooling2D((2,2))(x1)

x1 = Conv2D(256, (3,3), activation='relu', padding='same')(x1)
x1 = BatchNormalization()(x1)
x1 = MaxPooling2D((2,2))(x1)

x1 = GlobalAveragePooling2D()(x1)
x1 = Dense(128, activation='relu')(x1)
x1 = Dropout(0.3)(x1)

# --- MobileNetV2 Branch ---
base_mobilenet = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_mobilenet.trainable = False # Freeze initially

x2 = base_mobilenet(input_layer)
x2 = GlobalAveragePooling2D()(x2)
x2 = Dense(128, activation='relu')(x2)
x2 = Dropout(0.3)(x2)

# --- Concatenate Branches ---
merged = Concatenate()([x1, x2])
merged = Dense(256, activation='relu')(merged)
merged = BatchNormalization()(merged)
merged = Dropout(0.5)(merged)
merged = Dense(128, activation='relu')(merged)
merged = Dropout(0.3)(merged)
merged = Dense(64, activation='relu')(merged)

output_layer = Dense(7, activation='softmax')(merged)

model = Model(inputs=input_layer, outputs=output_layer)

# 3. Compile and Callbacks
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
    ReduceLROnPlateau(factor=0.2, patience=3, monitor='val_loss'),
    ModelCheckpoint('skin_cancer_hybrid_best.keras', save_best_only=True, monitor='val_loss')
]

# 4. Training Stage 1: Frozen MobileNetV2
print("Starting Training Stage 1...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=40,
    class_weight=class_weights_dict,
    callbacks=callbacks
)

# 5. Training Stage 2: Fine-tuning (Unfreeze top 30 layers)
print("\nStarting Training Stage 2 (Fine-tuning)...")
base_mobilenet.trainable = True
for layer in base_mobilenet.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_finetune = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    class_weight=class_weights_dict,
    callbacks=callbacks
)

# 6. Evaluation
def plot_history(history, history_finetune):
    acc = history.history['accuracy'] + history_finetune.history['accuracy']
    val_acc = history.history['val_accuracy'] + history_finetune.history['val_accuracy']
    loss = history.history['loss'] + history_finetune.history['loss']
    val_loss = history.history['val_loss'] + history_finetune.history['val_loss']
    
    epochs_range = range(len(acc))
    
    plt.figure(figsize=(15, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.title('Accuracy Graphs')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.title('Loss Graphs')
    plt.legend()
    plt.show()

plot_history(history, history_finetune)

# Test Accuracy & Report
test_loss, test_acc = model.evaluate(test_generator)
print(f"Test Accuracy: {test_acc*100:.2f}%")

Y_pred = model.predict(test_generator)
y_pred = np.argmax(Y_pred, axis=1)
y_true = test_generator.classes

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=encoder.classes_))

plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues', 
            xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# 7. Save Artifacts
model.save('skin_cancer_hybrid_model.h5')
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(encoder, f)

# 8. Grad-CAM and Single Image Prediction
def get_gradcam(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    
    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.reduce_max(heatmap)
    return heatmap.numpy()

def predict_and_visualize(img_path, model, encoder):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, (224, 224))
    img_array = np.expand_dims(img_resized, axis=0) / 255.0
    
    preds = model.predict(img_array)
    pred_idx = np.argmax(preds[0])
    pred_class = encoder.classes_[pred_idx]
    confidence = preds[0][pred_idx]
    
    print(f"Predicted: {pred_class} ({confidence*100:.2f}%)")
    print("Probabilities:", {c: f"{p*100:.2f}%" for c, p in zip(encoder.classes_, preds[0])})
    
    # Note: Using the last conv layer of the custom branch for Grad-CAM
    # Alternatively, you could use a layer from MobileNetV2
    try:
        # Finding the last Conv2D layer name
        conv_layers = [l.name for l in model.layers if isinstance(l, tf.keras.layers.Conv2D)]
        last_conv = conv_layers[-1]
        heatmap = get_gradcam(img_array, model, last_conv)
        
        heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
        heatmap = np.uint8(255 * heatmap)
        heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
        superimposed_img = cv2.addWeighted(img, 0.6, heatmap, 0.4, 0)
        
        plt.figure(figsize=(10, 5))
        plt.subplot(1, 2, 1)
        plt.imshow(img)
        plt.title('Original Image')
        plt.subplot(1, 2, 2)
        plt.imshow(superimposed_img)
        plt.title(f'Grad-CAM Heatmap: {pred_class}')
        plt.show()
    except Exception as e:
        print(f"Grad-CAM failed: {e}")
        plt.imshow(img)
        plt.title(f'Prediction: {pred_class}')
        plt.show()

# Example usage (uncomment and replace with actual path):
# predict_and_visualize(test_df.iloc[0]['path'], model, encoder)